# 第1回：Pythonとデータに触れ、まず予測を動かす

この回は3つのパートで構成します：**予測モデルを動かしてみる ／ Pythonを読み、Copilotと少し変える ／ pandasで表データに触る**。

**セルの動かし方**：各セル（灰色の枠）を選んで `Shift + Enter`（またはセル左の▷ボタン）を押すと実行できます。
**上から順に**実行してください。前のセルを飛ばすと、後のセルでエラーになります。

**AIと一緒に進める**：分からないコードは、セル全体ではなく気になる数行をM365 CopilotなどのAIへ貼って
説明や修正を相談します（`ASK COPILOT`）。ただしAIの答えは鵜呑みにせず、必ず自分の出力で確かめます。

`TRY`は全員、`CHANGE`は値を1つ変える練習、`CHALLENGE`は余裕がある人向け、
`DEEP DIVE`・`APPENDIX`は発展です（飛ばしても本編は完結します）。


In [ ]:
# 【準備セル】教材フォルダの場所を自動で見つけます。中身は今は理解しなくてOK、そのまま実行してください。
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

環境を整え、Pythonとpandasの基礎を身につけ、完成済みの予測モデルを動かして「学習と予測」を体で覚えます。

- 特徴量・目的変数・学習・予測を、画面上の入出力と結びつける
- 予測を関数へ切り出し、型ヒントとassertで最小の検証を付ける
- ベースラインと比べ、設定変更の効果を交差検証の平均とばらつきで語る
- 変数・リスト・辞書・条件分岐・繰り返し・関数を読める
- 型ヒント・docstring・防御的な入力検査を備えた関数を書く
- assertによる小さなテストで、境界値と例外を先に固定する
- 初見データの形・型・欠損・要約統計を確認する
- locとqueryで条件を明示し、method chainingで読みやすくまとめる
- groupby・agg・pivot_tableで多軸の比較表を作り、性能差にも気を配る

### この回の進み方（大切）

この回は、旧カリキュラムの**3回分をまとめた長い回**です。**パート1→2→3**の順に、各パートの
`CORE`（本線）で手を動かします。1回の時間で全部を終える必要はありません。各パートの
`DEEP DIVE`／`APPENDIX`は、余裕のある人や自習で進めてください。日をまたいで少しずつでも大丈夫です。

### 先に押さえる言葉

- 特徴量：予測時点でモデルへ渡す情報
- 目的変数：予測したい答え
- 学習：既知データから関係を推定する処理
- 予測：学習済みモデルを未知データへ使う処理
- 型ヒント：引数と戻り値の型を明示する注釈
- docstring：関数の目的と使い方を書く文字列
- 例外：処理を続けられない理由を伝える仕組み
- 単体テスト：関数の入出力を自動で確かめる小さなコード
- 純粋関数：同じ入力へ常に同じ出力を返し副作用のない関数
- DataFrame：行と列を持つ表
- method chaining：中間変数を作らず処理をつなげる書き方
- ベクトル化：ループの代わりに列全体へ一括演算すること
- 集約：複数行を件数や平均などへまとめる処理
- カテゴリ型：取りうる値が限られる列の省メモリ表現

> **実行前の30秒予想**：各パートの問いに、今の言葉で仮の答えを書いてから始めます。


---

# パート1：予測モデルを動かしてみる

**このパートの問い：予測モデルは、データを受け取って何を返しているのか。**


## この回で押さえる4つの言葉

この回は、モデルの中身のアルゴリズムは一旦置いて、**何を入れると何が返るか**だけを確認します。
モデルがやることは「過去のデータから関係を推定し（学習）、未知のデータへ当てはめる（予測）」の2段です。
先に、繰り返し出てくる4つの言葉を整理します。

| 言葉 | 意味 |
|---|---|
| 特徴量（X） | モデルへ渡す入力の列（例：分子量・LogP） |
| 目的変数（y） | 予測したい答えの列（例：活性 0/1） |
| 学習（fit） | 既知データから関係を推定する処理 |
| 予測（predict） | 学習済みモデルを未知データへ使う処理 |


## まずデータを開く

分析は「データを見る」ことから始まります。次のセルはCSV（表計算のような表データ）を読み込み、
`df`という名前の**表（DataFrame）**に入れます。`df.head()`は先頭5行だけを表示します。
全部で何行・何列あるかも一緒に出します。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


### 出力の読み方

- `420行 × 19列`：試料が420件、各試料について19種類の情報がある、という意味です。
- 表の**1行が1試料**、**1列が1種類の情報**です。`sample_id`は試料の名札で、予測には使いません。
- `NaN`（Not a Number）は**欠損＝その値が測られていない**印です。第3〜4回で詳しく扱います。

まだ意味が分からない列があっても大丈夫です。今日は下の5列だけ使います。


## モデルへ渡す列を決めて、学習させる

ここが今日の中心です。次のセルは4つの手順を続けて行っています。1行ずつ何をしているかは、
セルの下の「コードの読み方」で説明します。まず実行して、出てくる数字を眺めてください。

**なぜ「ベースライン」と比べるのか？** いきなり高機能なモデルの点数だけ見ても、それが
「すごい」のか「当たり前」なのか分かりません。そこで、**いつも多数派（ここでは非活性）と
答えるだけの単純なモデル**を先に用意し、本命がそれをどれだけ上回るかで価値を測ります。


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

features = ["molecular_weight", "logp", "tpsa", "h_bond_donors", "rotatable_bonds"]
X = df[features].fillna(df[features].median())
y = df["active"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
model = RandomForestClassifier(n_estimators=200, max_depth=4, random_state=42).fit(X_train, y_train)
for name, estimator in {"多数派ベースライン": baseline, "Random Forest": model}.items():
    pred = estimator.predict(X_valid)
    print(f"{name:14s} accuracy={accuracy_score(y_valid, pred):.3f}  F1={f1_score(y_valid, pred):.3f}")


### コードの読み方（1行ずつ）

- `features = [...]`：モデルへ渡す**入力列の名前リスト**。ここでは分子の性質5つを選びました。
- `X = df[features].fillna(...)`：`X`は入力の表。`fillna(...median())`は、欠損を**その列の中央値で埋める**処理です（モデルは空欄を扱えないため）。※この回では簡単のため**分割の前**に補完していますが、本来は分割の後に行うべきで、その理由は第6回で学びます。
- `y = df["active"]`：`y`は答えの列（活性=1／非活性=0）。
- `train_test_split(...)`：データを**学習用（train）と検証用（valid）に分ける**関数。`test_size=0.25`で25%を検証用に取り置きます。未知データでの成績を測るため、検証用は学習に使いません。`random_state=42`は分け方を固定して**毎回同じ結果**にするための指定、`stratify=y`は活性の割合が両側で揃うようにする指定です。
- `.fit(X_train, y_train)`：**学習**。過去データ（train）から関係を覚えます。
- `.predict(X_valid)`：覚えた関係を**検証用の未知データ**へ当てはめて予測します。
- `print(f"...{accuracy_score(...):.3f}...")`：`f"..."`は文字と計算結果を混ぜて表示する書き方（f文字列）。`{値:.3f}`は**小数第3位まで**表示する指定です。桁を変えたいときはこの数字を変えます。

### 出力の読み方

- **accuracy（正解率）**：全体のうち何割を当てたか。
- **F1**：活性を「見つける力」と「間違えない力」のバランス（0〜1、高いほど良い）。活性が少ないデータでは正解率より頼りになります（第8回で詳説）。
- 見るべきは**Random Forestがベースラインをどれだけ上回ったか**。差が小さいなら、そのモデルはまだ価値を出せていません。


## TRY：たった1試料を予測させてみる

モデルは表全体だけでなく、**1件ずつ**予測できます。検証用データの先頭1件を渡してみましょう。
`predict`は0か1の**判定**を、`predict_proba`は**活性である確率**を返します。


In [ ]:
one_sample = X_valid.iloc[[0]]
display(one_sample)
print("予測クラス:", model.predict(one_sample)[0])
print("活性である確率:", round(model.predict_proba(one_sample)[0, 1], 3))


### 出力の読み方と、よくある勘違い

- 上の表がこの試料の**入力（特徴量）**、その下がモデルの**答え**です。
- **予測クラス**が`1`なら「活性ありと判定」、`0`なら「非活性と判定」。
- **確率0.8**は「80%の確信で活性」という**モデルの自信**であって、「必ず活性」という保証ではありません。ここを混同しないことが、今日いちばん大事な感覚です。
- `iloc[[0]]`と二重角括弧にしているのは、1行でも**表の形のまま**渡すためです（`iloc[0]`だと1次元になり、モデルが受け取れません）。


## CORE深掘り：同じ評価は「関数」にまとめる

上では `accuracy_score(...)` と `f1_score(...)` を手で並べました。同じ評価を何度も書くと、
書き間違いが起きます。そこで**名前を付けた処理のかたまり（関数）**にまとめます。

- `def evaluate_classifier(...) -> dict:` の `-> dict` は「この関数は辞書を返す」という**型ヒント**（読み手への注釈）。
- 関数の1行目の文字列は**docstring**で、何をする関数かの説明です。
- `assert 条件, "メッセージ"` は「この条件が成り立たなければ止まれ」という**自己点検**。想定外の値が返っていないかを自動で見張ります。


In [ ]:
def evaluate_classifier(estimator, X_valid, y_valid) -> dict:
    "検証データでaccuracyとF1を計算し、辞書で返す純粋な評価関数。"
    pred = estimator.predict(X_valid)
    return {
        "accuracy": round(accuracy_score(y_valid, pred), 3),
        "f1": round(f1_score(y_valid, pred), 3),
    }

scores = evaluate_classifier(model, X_valid, y_valid)
assert set(scores) == {"accuracy", "f1"}, "返す指標が想定と違います"
assert 0.0 <= scores["f1"] <= 1.0, "F1は0〜1のはず"
scores


### なぜ関数にすると良いのか

- **繰り返しに強い**：別のモデルを評価したいとき、`evaluate_classifier(別のモデル, ...)`と呼ぶだけ。
- **間違いに気づける**：`assert`があるので、うっかりF1が1.2のような有り得ない値になったら即座に止まります。
- **読みやすい**：中身を知らなくても関数名で「何をするか」が伝わります。

この「小さく作って、テストで守る」考え方は第2回でさらに練習します。


## CHANGE：1か所だけ変えて、違いを観察する

`max_depth=4`（木の深さ）を`2`や`8`に変えて、上のセルを再実行してみましょう。
深くすると学習データには合いますが、検証スコアは必ずしも上がりません（**過学習**）。
変えた値・理由・結果を1行でメモしておきます。

## ASK COPILOT

M365 Copilotに、`fit`と`predict_proba`の違いを初心者向けに説明してもらいましょう。
返答を鵜呑みにせず、上の出力と照らして確かめます。

## まとめ

- 表の1行＝1試料、列＝情報。**特徴量（X）**を入れ、**目的変数（y）**を予測する。
- **fit=学習、predict=予測**。確率は「自信」であって真実ではない。
- 良し悪しは**ベースラインとの差**で測り、評価は**関数**にまとめて再利用する。


## DEEP DIVE：木の深さと「過学習」を交差検証で見る

ここからは経験者・自習向けの発展です。1回の学習/検証の分け方だと、たまたま簡単な検証データに
当たって点数が良く見えることがあります。そこで**交差検証**を使います。

**交差検証（cross validation）とは**：データを5つに分け、「4つで学習→残り1つで検証」を
担当を変えて5回行い、5回のスコアを平均する方法です。1回だけの運・不運をならして、
より信頼できる成績を出します。

次の表では、木の深さ（`max_depth`）を変えながら、**学習F1**と**検証F1**の両方を出します。
学習F1だけが高くて検証F1が伸びない＝**過学習**（覚えすぎて未知に弱い）のサインです。


In [ ]:
import pandas as pd
from sklearn.model_selection import cross_validate, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for depth in [1, 2, 3, 4, 6, 8, None]:
    estimator = RandomForestClassifier(n_estimators=200, max_depth=depth, random_state=42)
    result = cross_validate(estimator, X, y, cv=cv, scoring="f1", return_train_score=True)
    rows.append({
        "max_depth": str(depth),
        "学習F1": result["train_score"].mean(),
        "検証F1": result["test_score"].mean(),
        "検証F1_SD": result["test_score"].std(),
    })
pd.DataFrame(rows).round(3)


### 出力の読み方

- 上から下へ木を深くすると、**学習F1はほぼ単調に上がる**はずです（覚える力が増えるため）。
- 一方**検証F1**はどこかで頭打ち・悪化します。その手前が「ちょうど良い深さ」の目安です。
- `検証F1_SD`は5回のばらつき。小さいほど安定。**平均が少し高くてもSDが大きいモデル**は、運任せに近いので注意します。


### 特徴量重要度は2種類を見比べる

「どの特徴量が効いているか」を知りたくなります。ただし木モデルが標準で出す**不純度重要度**は、
値の種類が多い列を過大評価する癖があります。そこで、**列の値をわざと混ぜて性能がどれだけ落ちるか**で
測る**並べ替え重要度（permutation importance）**と並べて読みます。落ち幅が大きい列ほど本当に効いています。


In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(model, X_valid, y_valid, scoring="f1", n_repeats=20, random_state=42)
importance = pd.DataFrame({
    "特徴量": features,
    "不純度重要度": model.feature_importances_,
    "並べ替え重要度": perm.importances_mean,
    "並べ替えSD": perm.importances_std,
}).sort_values("並べ替え重要度", ascending=False)
importance.round(3)


### 出力の読み方

2つの列で順位が食い違ったら、**並べ替え重要度**を優先します。並べ替え重要度が0付近（SDより小さい）なら、
その特徴量は「効いているとは言い切れない」と読みます。


## CHALLENGE：確率は「当たっている」か（較正）

モデルが「確率0.8」と言った試料たちは、本当に約80%が活性でしょうか。確率を確率帯ごとに束ね、
**その帯の実際の活性率**と見比べます。予測確率と実際がだいたい一致していれば、確率を意思決定に
使えます（この一致度を**較正**と呼び、第8回で詳しく扱います）。


In [ ]:
probability = model.predict_proba(X_valid)[:, 1]
bucket = pd.cut(probability, bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0])
calibration = (
    pd.DataFrame({"確率帯": bucket, "実際の活性": y_valid.to_numpy()})
    .groupby("確率帯", observed=True)["実際の活性"]
    .agg(件数="size", 実際の活性率="mean")
)
calibration.round(3)


### 出力の読み方

各行は「その確率帯に入った試料の件数」と「実際に活性だった割合」です。
`0.6〜0.8`の帯で実際の活性率が0.7前後なら、確率はよく較正されています。大きくずれていたら、
確率の数字を鵜呑みにせず、順位付け（どれを先に試すか）にとどめる使い方が安全です。
なお件数が少ない帯は割合が不安定なので、件数も一緒に見ます。


## APPENDIX（任意・追加演習）

ここから先は90分では扱いません。手を動かして深めたい人向けの追加コードです。飛ばして次回へ進んでも
問題ありません。まずは**複数モデルを1つの関数でまとめて比較**します（第10回の予告編）。


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

def compare_models(candidates, X_tr, y_tr, X_va, y_va) -> pd.DataFrame:
    "候補モデルを同じデータで学習し、accuracyとF1の表を返す。"
    out = []
    for name, est in candidates.items():
        est.fit(X_tr, y_tr)
        pred = est.predict(X_va)
        out.append({"モデル": name, "accuracy": accuracy_score(y_va, pred), "F1": f1_score(y_va, pred)})
    return pd.DataFrame(out).sort_values("F1", ascending=False).round(3)

candidates = {
    "多数派": DummyClassifier(strategy="most_frequent"),
    "ロジスティック回帰": make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000)),
    "決定木": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=4, random_state=42),
    "勾配ブースティング": HistGradientBoostingClassifier(max_iter=200, random_state=42),
}
compare_models(candidates, X_train, y_train, X_valid, y_valid)


### 出力の読み方

F1の高い順に5モデルが並びます。**最も複雑なモデルが必ず1位とは限らない**こと、そして
「多数派」を全モデルが上回っているかを確認します。この`compare_models`関数は、以降の回でも使い回せます。


### ROC曲線と混同行列を並べて見る

分類の性能を2つの図で確認します。**ROC曲線**は閾値を動かしたときの当たり方を1本の曲線にしたもので、
曲線下の面積（AUC）が1に近いほど良い。**混同行列**は0.5で判定したときの内訳です。


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
RocCurveDisplay.from_estimator(model, X_valid, y_valid, ax=axes[0])
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set_title("ROC曲線（Random Forest）")
ConfusionMatrixDisplay.from_estimator(model, X_valid, y_valid, display_labels=["非活性", "活性"], cmap="Blues", ax=axes[1])
axes[1].set_title("混同行列（閾値0.5）")
plt.tight_layout()


### 出力の読み方

- **ROC曲線**：左上に張り付くほど良く、対角線（点線）はランダム予測。凡例のAUCが目安の数字です。
- **混同行列**：右下（活性を活性と当てた数）と、左下（活性を見逃した数）に注目。第8回で本格的に読み解きます。
- 指標の詳しい意味は第8回で扱うので、ここでは「こういう図で確認できる」と体験できれば十分です。


---

# パート2：Pythonを読み、Copilotと少し変える

**このパートの問い：分からないコードを、どうやって小さく理解し、安全に書き換えるか。**


## なぜ「読む」練習から始めるのか

これからの回では、完成したコードを**少しだけ書き換えて**実験します。ゼロから書けなくても、
**読めて・1か所いじれる**ようになれば十分に前へ進めます。この回は、以後ずっと出てくる
4つの部品（値・リスト・辞書・繰り返し・関数）に絞って読み方を身につけます。文法の網羅はしません。

Copilotは「一発で完成品を作らせる道具」ではなく、「短い相談を何度もする相棒」として使います。
提案は必ず1つずつ試し、出力を自分の目で確かめます。


## 変数・リスト・辞書：データを入れる3つの箱

- **変数**：1つの値に名前を付けた箱（例：`sample_name`）。
- **リスト `[...]`**：順番のある複数の値（例：温度の並び）。
- **辞書 `{key: value}`**：名前で値を引く箱（例：`experiment["solvent"]`で溶媒を取り出す）。

`type(x)`は「その値が何型か」を教えてくれます。実行して、3つの箱の見た目の違いを確かめましょう。


In [ ]:
sample_name = "CMP-0001"
temperatures = [60, 75, 90]
experiment = {"sample_id": sample_name, "solvent": "EtOH", "active": 1}
print(type(sample_name), sample_name)
print(type(temperatures), temperatures)
print(type(experiment), experiment)


### 出力の読み方

- `<class 'str'>`は**文字列**、`<class 'list'>`は**リスト**、`<class 'dict'>`は**辞書**。
- 辞書は`{'sample_id': 'CMP-0001', ...}`のように、**名前（キー）と値**の組で並びます。
- pandasの表（`df`）は、ざっくり言うと「辞書（列名→列の値）」と「リスト（行の並び）」を合わせたものです。この3つが分かると表データも読みやすくなります。


## TRY：`for`（繰り返し）と`if`（条件分岐）を読む

`for`は「リストの要素を1つずつ取り出して同じ処理を繰り返す」書き方、`if ... else`は
「条件で処理を分ける」書き方です。**実行する前に、何行表示されるか予想**してから動かしましょう。
予想と結果を比べるのが、コードを読む力を最短で伸ばすコツです。


In [ ]:
for temperature in temperatures:
    label = "高温条件" if temperature >= 75 else "低温条件"
    print(temperature, label)


### 出力の読み方

- `temperatures`は3要素なので**3行**出ます（予想は合っていましたか？）。
- 各行で`temperature`が60→75→90と変わり、`75以上か`で「高温／低温」が切り替わります。
- `A if 条件 else B`は「条件が真ならA、偽ならB」を1行で書く形。`if:` を複数行で書いても同じ意味です。


## 関数：処理に名前を付けて再利用する

**関数**は「入力を受け取り、決まった処理をして、結果を返す」部品です。同じ計算を何度も書かずに済みます。

- `def 関数名(引数: 型) -> 戻り値の型:` の**型ヒント**は、読み手（と生成AI）への注釈です。動作は変えませんが、誤解を減らします。
- 直後の文字列は**docstring**（関数の説明）。`help(関数)`で読めます。
- `[celsius_to_kelvin(v) for v in temperatures]`は**リスト内包表記**。「各要素に関数をかけた新しいリスト」を1行で作ります。


In [ ]:
def celsius_to_kelvin(celsius: float) -> float:
    "摂氏をケルビンへ変換する。"
    return celsius + 273.15

converted = [celsius_to_kelvin(value) for value in temperatures]
print(converted)
help(celsius_to_kelvin)


### 出力の読み方

- `converted`は、各温度に273.15を足したリスト（例：`[333.15, 348.15, 363.15]`）。
- `help(...)`は、書いておいたdocstringと引数の形を表示します。**自分の関数にも説明が付く**ことを体験しておきましょう。


## TRY：エラーは「読む」もの。省略せず全文を見る

エラーは失敗ではなく、**どこで何が起きたかの手がかり**です。わざと存在しない要素を取り出して、
エラーの形を観察します。`try/except`は「エラーが出ても止まらず、内容を受け取る」書き方です。


In [ ]:
try:
    temperatures[10]
except Exception as error:
    print(type(error).__name__)
    print(error)


### 出力の読み方

- `IndexError`という**エラーの種類（名前）**と、`list index out of range`という**説明**が出ます。
- リストは0番から数えるので、3要素の`temperatures`に`[10]`は存在せず、範囲外エラーになります。
- 実際のエラーでは、**末尾の1〜2行**（種類とメッセージ）にいちばん近い原因が書かれています。Copilotに貼るときも、この全文を省略しないことが大切です。


## CHANGE

`temperatures`へ温度を1つ追加し、`for`ループと変換結果の表示がどう変わるか確認します。

## ASK COPILOT

気になるセルを貼り、「各行の実行後に、変数の型と中身がどう変わるか表で説明して」と依頼します。
提案は1つずつ試し、必ず出力で答え合わせをします。


## DEEP DIVE：テストで守る小さなユーティリティ

ここからは発展です。「実行できる」ことと「正しい」ことは別物です。特にCopilotが書いたコードは、
**普通の入力では動いても、変な入力で静かに間違える**ことがあります。そこで、**変な入力を先に想定して
弾く関数**を書きます。

- `raise TypeError(...)` / `raise ValueError(...)` は、「この入力は受け付けない」と**わざとエラーを起こす**書き方。
- こうしておくと、間違った使い方をした人にすぐ気づいてもらえます（沈黙して誤った答えを返すより安全）。


In [ ]:
def celsius_to_kelvin_checked(celsius: float) -> float:
    "型と物理的な下限を検査してから摂氏をケルビンへ変換する。"
    if not isinstance(celsius, (int, float)):
        raise TypeError("温度は数値で入力してください")
    if celsius < -273.15:
        raise ValueError("絶対零度より低い値は指定できません")
    return celsius + 273.15

for value in [25, -273.15, -300, "25"]:
    try:
        print(value, "->", round(celsius_to_kelvin_checked(value), 2))
    except (TypeError, ValueError) as error:
        print(value, "->", type(error).__name__, error)


### 出力の読み方

4つの入力それぞれの結果が並びます。`25`は正常変換、`-273.15`は境界（絶対零度ちょうど）でOK、
`-300`は物理的にありえないので`ValueError`、`"25"`は数値でなく文字列なので`TypeError`。
**正常・境界・異常**を1度に確かめられました。


### assertで「期待する答え」を先に書いて固定する

`assert 式` は「式が真でなければ止まれ」という自己点検でした。これを使うと、関数の**テスト**が書けます。
コツは、**答えを先に書いてから**関数を作ること。ここではIQR法（四分位範囲）で外れ値を除く関数を、
正常・空リスト・NaN混在の3ケースで検証します。


In [ ]:
import numpy as np

def drop_outliers_iqr(values: list[float], k: float = 1.5) -> list[float]:
    "IQR法で外れ値を除いた値のリストを返す。NaNは事前に除く。"
    clean = [v for v in values if v == v]  # NaN(v != v)を除外
    if not clean:
        return []
    q1, q3 = np.percentile(clean, [25, 75])
    iqr = q3 - q1
    low, high = q1 - k * iqr, q3 + k * iqr
    return [v for v in clean if low <= v <= high]

assert drop_outliers_iqr([10, 11, 12, 13, 1000]) == [10, 11, 12, 13]
assert drop_outliers_iqr([]) == []
assert drop_outliers_iqr([5, 5, float("nan")]) == [5, 5]
print("すべてのテストを通過しました")


### 出力の読み方

- 3つの`assert`がすべて通ると、最後の`print`だけが表示されます。**エラーが出ない＝合格**です。
- もし関数を壊すと（例：`k`を0にする）、どの`assert`で止まったかが表示され、**間違いの場所がすぐ分かります**。
- `v == v`が`False`になるのはNaNだけ、という小技で欠損を除いています。


## CHALLENGE：関数の「性質」を調べる

外れ値を除いた後、もう一度同じ関数をかけると、さらに減るでしょうか。1回目で四分位が変わるため、
**必ずしも同じ結果（冪等）にはなりません**。こうした「関数の性質」を意識すると、思わぬ副作用に気づけます。


In [ ]:
rng = np.random.default_rng(0)
sample = rng.normal(50, 5, 200).tolist()
once = drop_outliers_iqr(sample)
twice = drop_outliers_iqr(once)
print("1回適用後の件数:", len(once))
print("2回目でさらに減った件数:", len(once) - len(twice))
print("2回目で変化なし(冪等):", once == twice)


### 出力の読み方

`2回目でさらに減った件数`が0でなければ、この関数は**冪等ではない**（適用回数で結果が変わる）と分かります。
外れ値除去を繰り返し適用する前処理は、この性質のせいで「消しすぎ」が起きやすい、という教訓につながります。


## APPENDIX（任意・追加演習）

90分の外で、以後よく使うPythonの部品をもう少し練習します。飛ばしても本編は進められます。
まずは`enumerate`（番号付き繰り返し）・`zip`（同時に回す）・`sorted`（並べ替え）・条件付き内包表記です。


In [ ]:
samples = ["CMP-0001", "CMP-0002", "CMP-0003"]
yields = [82.5, 40.1, 63.7]

for i, name in enumerate(samples, start=1):        # 番号付きで回す
    print(i, name)

pairs = {name: y for name, y in zip(samples, yields)}   # 2つを同時に回して辞書化
print("辞書:", pairs)

ranked = sorted(pairs.items(), key=lambda kv: kv[1], reverse=True)  # 収率降順
print("収率降順:", ranked)

high = [name for name, y in pairs.items() if y >= 60]   # 条件付き内包表記
print("収率60以上:", high)


### 出力の読み方

- `enumerate`は`(番号, 要素)`を返すので、行番号付きの表示に便利。
- `zip`は複数リストを同時に回します。`for a, b in zip(...)`の形は頻出です。
- `sorted(..., key=..., reverse=True)`で並べ替え。`key`に「何で並べるか」を関数で渡します。
- これらは`for`ループを短く読みやすくする道具で、pandasの内部でも同じ発想が使われています。


### 自分の「状態を持つ部品」を作る（クラス入門）

関数は入力→出力の1回きりですが、**クラス**は状態を持ち続けられます。値を足しながら件数と平均を
保つ小さなクラスを書き、`assert`で動作を確かめます。難しければ「こういう書き方がある」で十分です。


In [ ]:
class RunningStats:
    "値を1つずつ足しながら件数・合計・平均を保つ小さなクラス。"
    def __init__(self):
        self.n = 0
        self.total = 0.0
    def add(self, value: float) -> None:
        self.n += 1
        self.total += value
    @property
    def mean(self) -> float:
        return self.total / self.n if self.n else float("nan")

stats = RunningStats()
for y in [82.5, 40.1, 63.7]:
    stats.add(y)
assert stats.n == 3
assert abs(stats.mean - 62.1) < 0.1
print(f"件数={stats.n} 平均={stats.mean:.1f}")


### 出力の読み方

`add`を呼ぶたびに内部の`n`と`total`が更新され、`mean`はいつでも現在の平均を返します。`assert`が通れば
実装は期待どおり。scikit-learnのモデルも「`fit`で状態を覚え、`predict`で使う」クラスなので、この
仕組みが分かると内部のイメージがつかめます。


### pandasに橋渡しする

第3回で本格的に使うpandasを、ひと足先に少しだけ触ります。CSVを読み、1列（Series）の平均や
種類を取り出します。


In [ ]:
import pandas as pd

data = pd.read_csv(DATA / "compound_experiments.csv")
print("1列の型:", type(data["yield_pct"]).__name__)
print("平均収率:", round(data["yield_pct"].mean(), 1))
print("溶媒の種類:", data["solvent"].dropna().unique().tolist())


### 出力の読み方

`data["yield_pct"]`は1列（Series）で、`.mean()`のような集計をそのまま呼べます。`.unique()`は値の種類、
`.dropna()`は欠損を除く指定。ここまで来れば、第3回のpandasはぐっと読みやすくなります。


---

# パート3：pandasで表データに触る

**このパートの問い：初めて見る表データを受け取ったら、最初に何を見るか。**


## pandasは「表を操る道具」

pandasは、Excelのような表（DataFrame）をPythonで扱うライブラリです。研究データの多くは表なので、
これが読めると分析の8割は前に進みます。この回で身につけるのは、初見の表に対して**同じ手順で
最初の点検をする**習慣です。

初見データを受け取ったら、まず次の4つを見ます：**大きさ（行数×列数）／型（数値か文字か）／
欠損（空欄はどこか）／ばらつき（平均や範囲）**。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


## TRY：表の「健康診断」を1度に行う

次のセルは、点検の4項目をまとめて表示します。`df.shape`で大きさ、`df.dtypes`で型、
`df.isna()`で欠損、`df.describe()`で要約統計。**関数名がそのまま意味**なので、少しずつ覚えられます。


In [ ]:
print("形:", df.shape)
quality = pd.DataFrame({
    "データ型": df.dtypes.astype(str),
    "欠損数": df.isna().sum(),
    "欠損率": df.isna().mean().round(3),
    "ユニーク数": df.nunique(),
})
display(quality)
display(df.select_dtypes(include="number").describe().T.round(2))


### 出力の読み方

- **1つ目の表（品質表）**：各列の型・欠損数・欠損率・値の種類数。`object`は文字列、`float64`/`int64`は数値。欠損率が高い列や、`sample_id`のようにユニーク数＝行数の列（＝ただの名札）に気づけます。
- **2つ目の表（describe）**：数値列の件数・平均・標準偏差・最小/四分位/最大。`temperature_c`の`max`が極端に大きいなど、**怪しい値の当たり**をここで付けます。
- `.T`は表を**転置**（行列入れ替え）して、列がたくさんあっても縦に読めるようにする工夫です。


## 行と列を選ぶ：`loc` と `query`

分析は「必要な部分だけ取り出す」の連続です。2つの基本を覚えます。

- `df.loc[行の条件, 列のリスト]`：**場所を指定して取り出す**。
- `df.query("条件式")`：**条件を文章のように書いて絞り込む**。複数条件（`and`/`or`）が読みやすいのが利点です。


In [ ]:
columns = ["sample_id", "solvent", "catalyst", "temperature_c", "yield_pct", "active"]
display(df.loc[:4, columns])
subset = df.query("catalyst == 'Cat-A' and temperature_c >= 80")[columns]
print("Cat-Aかつ80℃以上:", len(subset), "件")
subset.head()


### 出力の読み方とつまずきポイント

- `df.loc[:4, columns]`は「行番号0〜4」×「指定した6列」。`loc`の範囲指定は**末尾を含む**点がPythonの通常のスライス（末尾を含まない）と違うので注意します。
- `query`の中では、文字列は`'Cat-A'`のように**引用符**で囲みます。列名はそのまま書けます。
- `len(subset)`で、条件に合った件数が分かります。**まず件数を確かめる**のは、絞り込みが意図どおりかの安全確認です。


## TRY：カテゴリごとにまとめて比べる（groupby）

「溶媒ごとの平均収率は？」のような問いには`groupby`が使えます。**同じ値の行をまとめて、
件数・平均・ばらつきなどを一気に計算**します。平均だけでなく**件数（size）とばらつき（std）**も
一緒に見るのが、だまされないコツです。


In [ ]:
solvent_summary = (
    df.groupby("solvent", dropna=False)
      .agg(件数=("sample_id", "size"), 平均収率=("yield_pct", "mean"),
           収率SD=("yield_pct", "std"), 活性率=("active", "mean"))
      .sort_values("平均収率", ascending=False)
)
solvent_summary.round(2)


### 出力の読み方

- 溶媒ごとに1行、件数・平均収率・収率のばらつき・活性率が並び、平均収率の高い順に並びます。
- **平均が高くても件数が極端に少ない**溶媒は、たまたまかもしれません。件数の小さい行の平均は割り引いて読みます。
- `dropna=False`にしているので、溶媒が欠損の行も1グループとして見えます（欠損を見逃さない工夫）。


## CHANGE

`groupby("solvent")`を`"catalyst"`や`"scaffold_group"`へ変えて、順位がどう変わるか見ます。
順位が変わる理由は、**データだけから断定せず仮説として**書き留めます（第4〜5回でその検証を学びます）。


## DEEP DIVE：多軸集計・処理の連結・速度

発展として、実務でよく使う3つを扱います。**pivot_table**（2軸のクロス集計）、
**pipe**（処理を関数でつなぐ）、そして**ベクトル化**（速く書く）です。


### pivot_table：2つの軸で同時に集計する

「触媒×溶媒」のように2軸で平均を見たいときは`pivot_table`が便利です。Excelのピボットテーブルと
同じ発想で、`index`（縦軸）・`columns`（横軸）・`values`（集計する値）・`aggfunc`（集計方法）を指定します。


In [ ]:
pivot = pd.pivot_table(df, index="catalyst", columns="solvent", values="yield_pct", aggfunc=["count", "mean"])
pivot.round(1)


### 出力の読み方

行が触媒、列が溶媒で、各マスに「件数」と「平均収率」が入ります。件数が0や極端に少ないマスは、
平均が空欄や不安定になります。**組み合わせによって効き方が変わる**様子（交互作用）の当たりを付けられます。


### pipe：処理を「関数の流れ」としてつなぐ

複数の加工を続けるとき、中間変数を増やすと読みにくくなります。`.pipe(関数)`を使うと、
**表を関数に通して次へ渡す**流れを、上から下へ素直に書けます。元データを壊さないよう、関数内で`copy()`します。


In [ ]:
def add_quality_flags(frame):
    "収率の中央値以上かどうかのフラグ列を足して返す（元は変更しない）。"
    out = frame.copy()
    out["high_yield"] = out["yield_pct"] >= out["yield_pct"].median()
    return out

summary = (
    df
    .pipe(add_quality_flags)
    .groupby(["catalyst", "high_yield"], observed=True)
    .agg(件数=("sample_id", "size"), 平均収率=("yield_pct", "mean"))
    .round(2)
)
summary


### 出力の読み方

触媒×「高収率かどうか」で件数と平均収率が出ます。**加工（フラグ付け）→集計**という流れが、
1つの縦長の式で読める点に注目してください。処理が増えても`.pipe(...)`を足すだけで拡張できます。


## CHALLENGE：`apply`と「ベクトル化」の速度差

同じ判定を2通りで書き、時間を比べます。行を1つずつ処理する`apply`は読みやすい一方、遅くなりがち。
列全体へ一括で演算する**ベクトル化**は速く、pandasの本領です。


In [ ]:
import time

def slow_flag(frame):
    return frame.apply(lambda r: r["temperature_c"] >= 80 and r["catalyst"] == "Cat-A", axis=1)

def fast_flag(frame):
    return (frame["temperature_c"] >= 80) & (frame["catalyst"] == "Cat-A")

t0 = time.perf_counter(); a = slow_flag(df); t1 = time.perf_counter()
b = fast_flag(df); t2 = time.perf_counter()
print("apply     :", round((t1 - t0) * 1000, 2), "ms")
print("vectorized:", round((t2 - t1) * 1000, 2), "ms")
print("結果一致:", bool((a.fillna(False) == b.fillna(False)).all()))


### 出力の読み方

- 2つの時間（ミリ秒）を比べると、**ベクトル化の方が速い**はずです。420行では差は小さくても、数十万行では体感が大きく変わります。
- `結果一致: True`は、2つの書き方が**同じ答え**を出した確認。速く書いても結果が同じであることを、必ず検証します。
- 教訓：`apply`が必要な場面もありますが、まず「列演算で書けないか」を考える習慣が、速く読みやすいコードにつながります。


## APPENDIX（任意・追加演習）

実データで頻出のpandas操作を、もう少し重めに練習します。90分の外の自習向けです。
まずは**度数の集計**（`value_counts`）と**クロス集計**（`crosstab`）です。


In [ ]:
print(df["catalyst"].value_counts(dropna=False))
print()
display(pd.crosstab(df["catalyst"], df["solvent"]))


### 出力の読み方

- `value_counts`は各カテゴリの件数を多い順に。`dropna=False`で欠損も1カテゴリとして数えます。
- `crosstab`は2つのカテゴリの組み合わせ件数の表。どの触媒×溶媒の組が多い／少ないかが一望できます。件数の少ない組は、後の分析で平均が不安定になりやすい箇所です。


### 日付を扱う（datetime）

`experiment_date`は文字列です。`pd.to_datetime`で日付型に変えると、月ごとの集計や期間の計算ができます。
時系列の分割（第6回）にもつながる大事な操作です。


In [ ]:
dated = df.copy()
dated["experiment_date"] = pd.to_datetime(dated["experiment_date"])
dated["month"] = dated["experiment_date"].dt.to_period("M").astype(str)
monthly = dated.groupby("month").agg(件数=("sample_id", "size"), 平均収率=("yield_pct", "mean")).round(1)
display(monthly.head(6))


### 出力の読み方

月ごとの件数と平均収率が並びます。`.dt.to_period("M")`で「年月」に丸めています。実データでは、
月やバッチで性能が変わることがあり、こうした時間軸の集計が異常検知や分割設計の入口になります。


### 表をつなぐ（merge）と、形を変える（melt）

`merge`は2つの表をキーで結合します。ここでは「触媒ごとの平均収率」を各行に付け直し、
各試料が平均より上か下かを計算します。`melt`は横広の表を縦長へ変える操作です。


In [ ]:
group_mean = df.groupby("catalyst")["yield_pct"].mean().rename("触媒平均収率").reset_index()
merged = df[["sample_id", "catalyst", "yield_pct"]].merge(group_mean, on="catalyst")
merged["平均との差"] = (merged["yield_pct"] - merged["触媒平均収率"]).round(1)
display(merged.head())

wide = df.head(3)[["sample_id", "molecular_weight", "logp", "tpsa"]]
long = wide.melt(id_vars="sample_id", var_name="記述子", value_name="値")
display(long)


### 出力の読み方

- **merge後**：各試料に「触媒平均収率」列が付き、「平均との差」で相対評価ができます。この「群平均を特徴量にする」発想は第11回のtarget encodingにつながります（ただしリークに注意）。
- **melt後**：3列だった記述子が「記述子・値」の2列に畳まれ、行数が増えます。可視化ライブラリはこの縦長形式を好むことが多いです。


---

## よくある誤り

- 学習データの成績を実力だと思う
- 1試料の予測だけでモデル全体を判断する
- 良い数値が出るまで設定を無計画に変える
- Notebookを途中から実行して変数がない
- Copilotの長い修正を一度に採用する
- エラー全文を読まずにセルを繰り返し実行する
- 列の単位や定義を確認せず計算する
- 行ごとのapplyを多用して遅く読みにくくする
- 件数が極端に少ない群の平均を強く信じる

## SELF-STUDY（任意・30〜60分）

- evaluate_classifierを拡張し、precisionとrecallも返してテストを足す
- 木の深さ2・4・8を交差検証で比較し、選ぶ理由を平均とばらつきで2文書く
- 収率のリストから外れ値をIQRで除く関数を、型ヒントとテスト付きで書く
- 正常値・空リスト・NaN混在の3ケースを、期待結果を先に書いてから検証する
- 触媒×溶媒の件数・平均収率・標準偏差をpivot_tableで作る
- applyとベクトル化の実行時間を比較し、差をm%で記録する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. Xとyはそれぞれ何か
2. 不純度重要度と並べ替え重要度はどう違うか
3. 単一の検証スコアより交差検証を見る理由は何か
4. 型ヒントとdocstringは何の役に立つか
5. assertは何を保証し、何を保証しないか
6. 生成AIのコードを何で確認するか
7. shapeの2つの数は何か
8. method chainingの利点と注意点は何か
9. applyよりベクトル化を選ぶ理由は何か

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
